# Clustering Mental Health Features

This analysis clusters haunted locations based on mental health distress, reported supernatural events, and apparition types. The features used for clustering are:

* Average_Mental_Health_Days: The average number of days per 30 days individuals in the area reported experiencing poor mental health.
* Average_Poor_Health_Days: The average number of days per 30 days individuals reported having either poor mental or physical health.
* Depression_Prevalence: The percentage of people in the area who have been diagnosed with depression.
* Event_Type: The category of supernatural or unexplained events associated with the haunted location (e.g., accident/disaster, supernatural occurrence, or violence).
* Apparition_Type: The type of supernatural entity reported (e.g., ghost, demon, spirit, orb).

By analyzing these factors, this clustering aims to investigate whether areas with higher mental health distress report more supernatural activity and whether certain event types and apparition reports correlate with different levels of psychological well-being.

# Generating Indices

The indices were chosen to ensure a balanced and representative sample of haunted places based on mental health distress levels, apparition type, and event type. First, 500 haunted places in areas with the highest mental health distress and 500 in areas with the lowest distress were selected to compare supernatural activity in regions with varying psychological well-being. This selection was based on three key indicators: "Average_Mental_Health_Days," "Average_Poor_Health_Days," and "Depression_Prevalence" to capture a holistic view of mental health conditions.

Then, a diverse sample of up to 100 hauntings per apparition type was included to analyze how different supernatural experiences may correlate with mental health distress. This step ensures that a broad range of apparitions, such as ghosts, demons, spirits, and orbs, are fairly represented in the dataset. Finally, hauntings were grouped by event type, with up to 50 samples per type randomly selected to prevent any one category from dominating the analysis. The final dataset combines these selections, allowing for meaningful clustering that examines the relationship between mental health distress, reported hauntings, and the nature of supernatural experiences.

In [ ]:
# Convert relevant columns to numeric types
df['Average_Mental_Health_Days'] = pd.to_numeric(df['Average_Mental_Health_Days'], errors='coerce')
df['Average_Poor_Health_Days'] = pd.to_numeric(df['Average_Poor_Health_Days'], errors='coerce')
df['Depression_Prevalence'] = pd.to_numeric(df['Depression_Prevalence'], errors='coerce')

# Number of rows to select per extreme group (adjust as needed)
num_rows_per_group = 500  

# Select haunted places in areas with the HIGHEST mental health distress cases
top_mental_health = df.sort_values(by=['Average_Mental_Health_Days', 'Average_Poor_Health_Days', 'Depression_Prevalence'], 
                                   ascending=[False, False, False]).head(num_rows_per_group)

# Select haunted places in areas with the LOWEST mental health distress cases
low_mental_health = df.sort_values(by=['Average_Mental_Health_Days', 'Average_Poor_Health_Days', 'Depression_Prevalence'], 
                                   ascending=[True, True, True]).head(num_rows_per_group)

# Get the index numbers of these rows
top_mental_indices = top_mental_health.index.tolist()
low_mental_indices = low_mental_health.index.tolist()

# Combine both sets of indices
selected_indices = top_mental_indices + low_mental_indices

# Select a subset of hauntings by apparition type
selected_hauntings = df.loc[selected_indices].groupby('Apparition_Type', group_keys=False).apply(lambda x: x.sample(n=min(100, len(x)), random_state=42))

# Get the indices of selected hauntings by apparition type (ensuring they are numeric)
apparition_indices = selected_hauntings.index.tolist()

# Select a subset of hauntings by event type
event_type_samples = selected_hauntings.groupby('Event_Type', group_keys=False).apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))

# Get the indices of selected hauntings by event type
event_type_indices = event_type_samples.index.tolist()

# Combine all selected indices
final_selected_indices = apparition_indices + event_type_indices

# # Print the final indices
# print("Selected Haunted Place Indices:", final_selected_indices)

# Jaccard Test

In [114]:
import pandas as pd
import os
import sys 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/jaccard/mental_health/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Jaccard Clusters

In [117]:
#Mental Health Distress Across Clusters
mental_health_summary = df.groupby('Cluster')[['Average_Mental_Health_Days', 
                                               'Average_Poor_Health_Days', 
                                               'Depression_Prevalence']].describe()
print("Mental Health Distress Across Clusters:\n", mental_health_summary)

#Apparition Types by Cluster
apparition_distribution = df.groupby(['Cluster', 'Apparition_Type']).size().unstack(fill_value=0)
print("\nApparition Type Distribution Across Clusters:\n", apparition_distribution)

# Normalize apparition type distribution per cluster
apparition_proportions = apparition_distribution.div(apparition_distribution.sum(axis=1), axis=0)
print("\nNormalized Apparition Type Proportions by Cluster:\n", apparition_proportions)

# Event Types by Cluster
event_distribution = df.groupby(['Cluster', 'Event_Type']).size().unstack(fill_value=0)
print("\nEvent Type Distribution Across Clusters:\n", event_distribution)

# Normalize event type distribution per cluster
event_proportions = event_distribution.div(event_distribution.sum(axis=1), axis=0)
print("\nNormalized Event Type Proportions by Cluster:\n", event_proportions)

Mental Health Distress Across Clusters:
           Average_Mental_Health_Days                                                   Average_Poor_Health_Days                                                    Depression_Prevalence                                                          
                               count      mean       std   min   25%   50%   75%   max                    count      mean       std   min   25%    50%   75%   max                 count      mean       std    min      25%     50%    75%    max
Cluster                                                                                                                                                                                                                                           
cluster 0                       12.0  4.403333  0.990467  3.29  3.72  3.76  5.51  5.51                     12.0  5.445833  0.730311  4.53  4.75  5.435  6.18  6.18                  12.0  0.217250  0.048932  0.163  0.18375  0.1850  

# Edit Distance

In [127]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/edit-distance/mental_health/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investiagting Edit Distance Clusters

In [130]:
#Mental Health Distress Across Clusters
mental_health_summary = df.groupby('Cluster')[['Average_Mental_Health_Days', 
                                               'Average_Poor_Health_Days', 
                                               'Depression_Prevalence']].describe()
print("Mental Health Distress Across Clusters:\n", mental_health_summary)

#Apparition Types by Cluster
apparition_distribution = df.groupby(['Cluster', 'Apparition_Type']).size().unstack(fill_value=0)
print("\nApparition Type Distribution Across Clusters:\n", apparition_distribution)

# Normalize apparition type distribution per cluster
apparition_proportions = apparition_distribution.div(apparition_distribution.sum(axis=1), axis=0)
print("\nNormalized Apparition Type Proportions by Cluster:\n", apparition_proportions)

# Event Types by Cluster
event_distribution = df.groupby(['Cluster', 'Event_Type']).size().unstack(fill_value=0)
print("\nEvent Type Distribution Across Clusters:\n", event_distribution)

# Normalize event type distribution per cluster
event_proportions = event_distribution.div(event_distribution.sum(axis=1), axis=0)
print("\nNormalized Event Type Proportions by Cluster:\n", event_proportions)

Mental Health Distress Across Clusters:
           Average_Mental_Health_Days                                                    Average_Poor_Health_Days                                                   Depression_Prevalence                                                         
                               count      mean       std   min   25%   50%    75%   max                    count      mean       std   min   25%   50%   75%   max                 count      mean       std    min     25%    50%     75%    max
Cluster                                                                                                                                                                                                                                          
cluster 0                        1.0  5.510000       NaN  5.51  5.51  5.51  5.510  5.51                      1.0  6.180000       NaN  6.18  6.18  6.18  6.18  6.18                   1.0  0.272000       NaN  0.272  0.2720  0.272  0.272

# Cosine

In [132]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/cosine/mental_health/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Cosine Clusters

In [134]:
#Mental Health Distress Across Clusters
mental_health_summary = df.groupby('Cluster')[['Average_Mental_Health_Days', 
                                               'Average_Poor_Health_Days', 
                                               'Depression_Prevalence']].describe()
print("Mental Health Distress Across Clusters:\n", mental_health_summary)

#Apparition Types by Cluster
apparition_distribution = df.groupby(['Cluster', 'Apparition_Type']).size().unstack(fill_value=0)
print("\nApparition Type Distribution Across Clusters:\n", apparition_distribution)

# Normalize apparition type distribution per cluster
apparition_proportions = apparition_distribution.div(apparition_distribution.sum(axis=1), axis=0)
print("\nNormalized Apparition Type Proportions by Cluster:\n", apparition_proportions)

# Event Types by Cluster
event_distribution = df.groupby(['Cluster', 'Event_Type']).size().unstack(fill_value=0)
print("\nEvent Type Distribution Across Clusters:\n", event_distribution)

# Normalize event type distribution per cluster
event_proportions = event_distribution.div(event_distribution.sum(axis=1), axis=0)
print("\nNormalized Event Type Proportions by Cluster:\n", event_proportions)

Mental Health Distress Across Clusters:
           Average_Mental_Health_Days                                                   Average_Poor_Health_Days                                                  Depression_Prevalence                                                     
                               count      mean       std   min   25%   50%   75%   max                    count      mean       std   min   25%   50%   75%  max                 count      mean     std    min    25%    50%    75%    max
Cluster                                                                                                                                                                                                                                    
cluster 0                      314.0  4.626879  0.922277  3.29  3.75  5.14  5.51  5.82                    314.0  5.687834  0.717917  4.53  5.02  5.61  6.18  6.9                 314.0  0.229197  0.0448  0.163  0.185  0.254  0.272  0.287

Apparition Typ